# 04 · Fotometría de superficie sintética — isofotas + Sérsic (método alternativo)

**Objetivo:** medir el radio de Holmberg (μ_B = 26.5) con un método independiente al de
`01_`-`01f_`, simulando el proceso "observacional" completo en vez de un perfil 1D directo
sobre el radio de cada partícula:

1. Se deposita la luminosidad de las partículas estelares del subhalo en una **imagen
   sintética 2D** (grilla + suavizado gaussiano, imitando ruido de discretización/PSF).
2. Se ajustan **isofotas elípticas** sobre esa imagen con `photutils.isophote`
   (método de Jedrzejewski 1987).
3. Se ajusta un **perfil de Sérsic** a μ_B(a).
4. Se estima el **radio de Holmberg de dos formas independientes**: interpolación empírica
   del perfil medido, y a partir del modelo de Sérsic — y se compara la diferencia.
5. Se chequea la **robustez**: dispersión entre proyecciones (xy/xz/yz) y convergencia con
   la escala de píxel y el suavizado.

**Diferencia respecto a `01_`-`01f_`:** esos notebooks miden μ(r) directamente desde el
radio 2D de cada partícula (sin pixelizar). Aquí se construye primero una imagen — más
parecido a cómo se mediría en datos observacionales reales.

**Nota sobre M_sun,B:** este pipeline usa **5.44** (valor de referencia para este método),
distinto del `M_SUN_B_VEGA = 5.36` (Willmer 2018) usado en `params_icl.py` para `01_`-`01f_`.
Se mantiene independiente para no alterar esos notebooks; ambos son valores de literatura
razonables para M_sun,B y la diferencia (~0.08 mag) es pequeña frente a la incertidumbre
del método.

**Dependencia nueva:** `photutils` (agregado a `environment.yml`).

In [1]:
import sys, os
sys.path.insert(0, '..')
sys.path.insert(0, '../original_shift_code')

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

import illustris_python as il
import Catalogue
import params_icl as P
import sb_isophote_pipeline as sbp

FIG_PDF = './figuras/pdf'
FIG_PNG = './figuras/png'
OUT_DIR = './outputs'
for d in (FIG_PDF, FIG_PNG, OUT_DIR):
    os.makedirs(d, exist_ok=True)

CATALOG_PATH = os.path.join('..', P.CATALOG_OUT)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110,
                     'font.size': 12,
                     'axes.spines.top': False,
                     'axes.spines.right': False})


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Selección del subhalo demo

Mismo cúmulo demo que `01_`-`01f_` (el más masivo, `i_demo=0`) para poder comparar
directamente el R_H obtenido aquí contra el de los notebooks anteriores.

In [2]:
with h5py.File(CATALOG_PATH, 'r') as f:
    group_idx   = f['group_idx'][:]
    M200c       = f['M200c_Msun'][:]
    R200c       = f['R200c_kpc'][:]
    GroupPos    = f['GroupPos_kpc'][:]
    bcg_sub_idx = f['bcg_sub_idx'][:]

i_demo = 0
sub_id = int(bcg_sub_idx[i_demo])
cen    = GroupPos[i_demo]

Header  = il.groupcat.loadHeader(P.basePath, P.SNAP)
box_kpc = Header['BoxSize'] * P.UL

print(f"Cúmulo demo: group_idx={group_idx[i_demo]}, sub_id={sub_id}")
print(f"log M200c = {np.log10(M200c[i_demo]):.2f}   R200c = {R200c[i_demo]:.1f} kpc")


NameError: name 'h5py' is not defined

## 1. Imagen sintética — carga, proyección y grilla

Se carga el subhalo central (BCG), se centra en su posición, y se corre el pipeline
completo (`run_projection`) para la proyección por defecto `xy` **sin** rotar a ejes
principales (`align_principal=False`) — la robustez frente a la orientación se chequea
más abajo con las 3 proyecciones xy/xz/yz.

In [ ]:
pos_c, mass, phot = sbp.load_subhalo_stars(
    il, P.basePath, P.SNAP, sub_id, cen, box_kpc,
    P.UL, P.UM, Catalogue.Distance_3D)

print(f"N partículas estelares del subhalo: {len(mass):,}")

result = sbp.run_projection(
    pos_c, mass, phot,
    projection='xy', align_principal=False,
    pixel_scale_kpc=0.2, smooth_px=1.5,
    band_idx=1, M_sun_band=sbp.M_SUN_B,
    min_particles=50)


## 2-5. Brillo superficial, isofotas, Sérsic y radio de Holmberg

`run_projection` ya encadena: imagen → μ_B(imagen) → ajuste de isofotas → ajuste de
Sérsic → radio de Holmberg (empírico y de modelo).

In [ ]:
print(f"Isofotas ajustadas       : {len(result['table'])}")
print(f"Isofotas confiables      : {result['table']['reliable'].sum()}  "
      f"(n_part >= 50 partículas)")
print()
print(f"Sérsic  mu_e = {result['mu_e']:.2f} ± {result['mu_e_err']:.2f} mag/arcsec^2")
print(f"Sérsic  Re   = {result['Re']:.2f} ± {result['Re_err']:.2f} kpc")
print(f"Sérsic  n    = {result['n']:.2f} ± {result['n_err']:.2f}")
print()
print(f"R_H empírico (interpolación) : {result['r_h_empirical']:.2f} kpc")
print(f"R_H modelo (Sérsic)          : {result['r_h_model']:.2f} kpc")
print(f"Δ R_H                        : {abs(result['r_h_empirical'] - result['r_h_model']):.2f} kpc")


### Figura — imagen sintética + isofotas, y perfil μ(a)

In [ ]:
fig = sbp.plot_galaxy_isophotes(result, sub_id, FIG_PDF, FIG_PNG)
plt.show()


## Guardar tabla de isofotas (CSV)

In [ ]:
csv_path = f"{OUT_DIR}/isophotes_sub{sub_id}_{result['projection']}.csv"
sbp.save_isophote_csv(result['table'], csv_path)
print(f"Guardado: {csv_path}")
result['table'].head(10)


## 6. Robustez — proyecciones xy / xz / yz

Se repite el pipeline completo en las 3 proyecciones cartesianas del box (sin rotar a
ejes principales) y se reporta la dispersión de R_H entre ellas — si el método es
robusto, R_H no debería depender fuertemente del ángulo de vista.

In [ ]:
projections = ['xy', 'xz', 'yz']
results_proj = {}
rows_proj = []

for proj in projections:
    res = sbp.run_projection(pos_c, mass, phot, projection=proj, align_principal=False,
                              pixel_scale_kpc=0.2, smooth_px=1.5, min_particles=50)
    results_proj[proj] = res
    sbp.save_isophote_csv(res['table'], f"{OUT_DIR}/isophotes_sub{sub_id}_{proj}.csv")
    sbp.plot_galaxy_isophotes(res, sub_id, FIG_PDF, FIG_PNG)
    plt.close()
    rows_proj.append(sbp.summary_row(res, sub_id))

df_proj = pd.DataFrame(rows_proj)
df_proj[['projection', 'mu_e', 'Re_kpc', 'n', 'R_H_empirical_kpc', 'R_H_model_kpc']]


In [ ]:
for col in ['R_H_empirical_kpc', 'R_H_model_kpc']:
    vals = df_proj[col].to_numpy()
    print(f"{col}: media={np.nanmean(vals):.2f} kpc  "
          f"std={np.nanstd(vals):.2f} kpc  "
          f"dispersión relativa={100*np.nanstd(vals)/np.nanmean(vals):.1f}%")


## 6. Robustez — convergencia con escala de píxel y suavizado

Se corre el pipeline variando `pixel_scale_kpc` y `smooth_px` (proyección xy) para
verificar que R_H no dependa fuertemente de estos parámetros de discretización.

In [ ]:
df_conv = sbp.convergence_grid(
    pos_c, mass, phot, projection='xy',
    pixel_scales=(0.1, 0.2, 0.3, 0.5), smooth_pxs=(1.0, 1.5, 2.0),
    min_particles=50)

df_conv.to_csv(f"{OUT_DIR}/convergencia_pixel_smooth_sub{sub_id}.csv", index=False)
df_conv


In [ ]:
for col in ['r_h_empirical', 'r_h_model']:
    vals = df_conv[col].dropna().to_numpy()
    if len(vals):
        print(f"{col}: media={np.mean(vals):.2f} kpc  std={np.std(vals):.2f} kpc  "
              f"dispersión relativa={100*np.std(vals)/np.mean(vals):.1f}%  (N={len(vals)} combinaciones)")


## 7. Tabla resumen final

Combina la corrida principal (`xy`, sin rotar) y las 3 proyecciones de robustez en una
sola tabla resumen con los parámetros de Sérsic y ambos radios de Holmberg.

In [ ]:
summary_rows = [sbp.summary_row(result, sub_id, extra={'run': 'principal'})]
summary_rows += [sbp.summary_row(results_proj[p], sub_id, extra={'run': 'robustez_proyeccion'})
                 for p in projections]

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(f"{OUT_DIR}/summary_sersic_holmberg_sub{sub_id}.csv", index=False)
df_summary
